In [6]:
#!/usr/bin/env python3
"""
=============================================================================
Padma River Morphodynamics Analysis Pipeline
=============================================================================
Implements the complete statistical and spatial analysis methodology:
  § 4.1 – Areal Dynamics and Seasonal Variability
  § 4.2 – Spatial Change Analysis: Erosion and Accretion Mapping
  § 4.3 – Morphological Metrics and Channel Migration
  § 4.4 – Probabilistic Mapping and Historical Frequency (WOF + Markov)

Data sources used:
  ├── Yearly     composites  (1988–2025, fully complete, 38 images)
  ├── Bi-monthly composites  (BM1–BM6, 156 usable images)
  └── Quarterly  composites  (Q1–Q4,   125 usable images)

NOTE: 1987 excluded – imagery confirmed broken/unusable.
      Study period begins 1988.

Water mask convention : 1 = Water, 0 = Land, 255/NaN = No-data
=============================================================================
"""

# ── Standard library ─────────────────────────────────────────────────────────
import os, glob, re, warnings
from pathlib import Path

# ── Third-party ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import Affine
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
from scipy.ndimage import (
    label as ndlabel,
    center_of_mass,
    distance_transform_edt,
    binary_fill_holes,
    gaussian_filter,
)
from skimage.morphology import (
    skeletonize,
    remove_small_objects,
    disk,
)
from skimage.measure import regionprops

warnings.filterwarnings("ignore")

# ── Optional: pymannkendall ───────────────────────────────────────────────────
try:
    import pymannkendall as mk
    MK_AVAILABLE = True
except ImportError:
    MK_AVAILABLE = False
    warnings.warn(
        "pymannkendall not found – Mann-Kendall will use scipy fallback.\n"
        "Install with:  pip install pymannkendall"
    )

# ── Optional: tqdm ────────────────────────────────────────────────────────────
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(x, **kw):
        return x

# ── Python < 3.10 fallback for itertools.pairwise ────────────────────────────
try:
    from itertools import pairwise
except ImportError:
    def pairwise(iterable):
        it = iter(iterable)
        a = next(it, None)
        for b in it:
            yield a, b
            a = b


# =============================================================================
# 0.  CONFIGURATION
# =============================================================================

class Config:
    """
    Central configuration block.
    Edit ONLY this class to adapt the pipeline to your directory layout.
    """

    # ── Input directories (Kaggle paths) ──────────────────────────────────────
    # Stored as Path objects so the / operator works everywhere.
    YEARLY_DIR    = Path("/kaggle/input/datasets/kaoserahamed/riverbank/river")
    BIMONTHLY_DIR = Path("/kaggle/input/datasets/kaoserahamed/bimonthly-filled-latest")
    QUARTERLY_DIR = Path("/kaggle/input/datasets/kaoserahamed/quarterlygapfilled")

    # ── Output directories ────────────────────────────────────────────────────
    OUTPUT_DIR  = Path("/kaggle/working/outputs")
    FIGURES_DIR = OUTPUT_DIR / "figures"
    CSV_DIR     = OUTPUT_DIR / "csv"
    RASTER_DIR  = OUTPUT_DIR / "rasters"

    # ── Study period ──────────────────────────────────────────────────────────
    STUDY_START = 1988
    STUDY_END   = 2025

    # ── Spatial parameters ────────────────────────────────────────────────────
    PIXEL_SIZE_M  = 60
    PIXEL_AREA_HA = (60 ** 2) / 10_000      # 0.36 ha / pixel

    # ── Transect parameters ───────────────────────────────────────────────────
    TRANSECT_SPACING_PX  = 17
    TRANSECT_HALF_LEN_PX = 250

    # ── Island / char-land tracking ───────────────────────────────────────────
    MIN_ISLAND_PX = 50

    # ── WOF zone thresholds (%) ───────────────────────────────────────────────
    WOF_PERMANENT  = 80
    WOF_ACTIVE_LOW = 10

    # ── Decadal boundaries ────────────────────────────────────────────────────
    DECADAL_BOUNDARIES = [1988, 1998, 2008, 2018, 2025]

    # ── Colour maps ───────────────────────────────────────────────────────────
    CMAP_WOF    = "RdYlBu"
    CMAP_CHANGE = mcolors.ListedColormap(
        ["#d73027",   # +1  Erosion   (Land → Water)
         "#f7f7f7",   #  0  Stable
         "#4575b4"]   # -1  Accretion (Water → Land)
    )

    # ── Figure sizes ──────────────────────────────────────────────────────────
    FIGSIZE_WIDE = (14, 5)
    FIGSIZE_MAP  = (10, 8)
    FIGSIZE_TALL = (12, 9)
    DPI          = 150

    # ── Bi-monthly period labels ──────────────────────────────────────────────
    BM_LABELS = {
        1: "Jan–Feb",
        2: "Mar–Apr",
        3: "May–Jun",
        4: "Jul–Aug",
        5: "Sep–Oct",
        6: "Nov–Dec",
    }


cfg = Config()

# ── Create all output directories ────────────────────────────────────────────
for _d in [cfg.FIGURES_DIR, cfg.CSV_DIR, cfg.RASTER_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi":        cfg.DPI,
    "font.family":       "DejaVu Sans",
    "font.size":         11,
    "axes.titlesize":    13,
    "axes.labelsize":    12,
    "legend.fontsize":   10,
    "figure.facecolor":  "white",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

print("=" * 65)
print("  Padma River Morphodynamics Pipeline  –  initialised")
print(f"  Study period : {cfg.STUDY_START}–{cfg.STUDY_END}  "
      f"(1987 excluded – broken imagery)")
print("=" * 65)


# =============================================================================
# 1.  DATA LOADING UTILITIES
# =============================================================================

def _parse_year(stem: str):
    m = re.search(r"(\d{4})", stem)
    return int(m.group(1)) if m else None

def _parse_quarter(stem: str):
    m = re.search(r"[Qq](\d)", stem)
    return int(m.group(1)) if m else None

def _parse_bimonth(stem: str):
    m = re.search(r"[Bb][Mm](\d)", stem)
    return int(m.group(1)) if m else None


def load_mask(path):
    """
    Load a binary GeoTIFF water mask.

    Returns
    -------
    mask      : bool ndarray (H, W),  True = water
    transform : rasterio Affine
    meta      : dict  {crs, transform, width, height, nodata}
    """
    with rasterio.open(path) as src:
        data   = src.read(1).astype(float)
        nodata = src.nodata if src.nodata is not None else 255
        data[data == nodata] = np.nan
        mask = (data == 1)
        return mask, src.transform, {
            "crs":       src.crs,
            "transform": src.transform,
            "width":     src.width,
            "height":    src.height,
            "nodata":    nodata,
        }


def save_raster(array, ref_meta, out_path, dtype="float32"):
    """Write a single-band GeoTIFF using a reference metadata dict."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # ── Pick a nodata value that is valid for the requested dtype ─────────────
    _dtype = np.dtype(dtype)
    if np.issubdtype(_dtype, np.floating):
        nodata = -9999.0
    elif np.issubdtype(_dtype, np.unsignedinteger):
        nodata = int(np.iinfo(_dtype).max)          # e.g. 255 for uint8
    else:                                            # signed integer
        nodata = int(np.iinfo(_dtype).min)          # e.g. -128 for int8

    profile = {
        "driver":    "GTiff",
        "dtype":     dtype,
        "width":     ref_meta["width"],
        "height":    ref_meta["height"],
        "count":     1,
        "crs":       ref_meta["crs"],
        "transform": ref_meta["transform"],
        "nodata":    nodata,
        "compress":  "lzw",
    }
    with rasterio.open(out_path, "w", **profile) as dst:
        arr = array.astype(float)
        arr[np.isnan(arr)] = nodata
        dst.write(arr.astype(dtype), 1)


def pixel_area_ha(mask):
    """Total water-pixel count → hectares."""
    return float(np.nansum(mask)) * cfg.PIXEL_AREA_HA


def _shape_from_catalog(catalog):
    """Return (H, W) by peeking at the first file in the catalog."""
    with rasterio.open(catalog.iloc[0]["path"]) as src:
        return src.height, src.width


# ── Catalog builders ──────────────────────────────────────────────────────────

def build_yearly_catalog(directory):
    """
    Scan directory for GeoTIFFs and return a sorted DataFrame.
    Accepts str or Path; years outside [STUDY_START, STUDY_END] are dropped.
    """
    directory = Path(directory)          # ← ensures Path object regardless
    files     = sorted(glob.glob(str(directory / "*.tif")))
    records   = []
    skipped   = []

    for f in files:
        year = _parse_year(Path(f).stem)
        if year is None:
            continue
        if year < cfg.STUDY_START:
            skipped.append(year)
            continue
        if year > cfg.STUDY_END:
            continue
        records.append({"year": year, "path": f})

    df = (pd.DataFrame(records)
            .sort_values("year")
            .reset_index(drop=True))

    if df.empty:
        print(f"  [Yearly]     0 files found in {directory}")
        return df

    print(f"  [Yearly]     {len(df):3d} files  "
          f"({df.year.min()}–{df.year.max()})")
    if skipped:
        print(f"               Skipped years before {cfg.STUDY_START}: "
              f"{sorted(skipped)}")
    return df


def build_bimonthly_catalog(directory):
    directory = Path(directory)          # ← same fix
    files     = sorted(glob.glob(str(directory / "*.tif")))
    records   = []

    for f in files:
        stem = Path(f).stem
        year = _parse_year(stem)
        bm   = _parse_bimonth(stem)
        if (year is None or bm is None
                or year < cfg.STUDY_START
                or year > cfg.STUDY_END):
            continue
        records.append({
            "year":    year,
            "bimonth": bm,
            "label":   cfg.BM_LABELS.get(bm, f"BM{bm}"),
            "period":  f"{year}_BM{bm}",
            "path":    f,
        })

    df = (pd.DataFrame(records)
            .sort_values(["year", "bimonth"])
            .reset_index(drop=True))

    if df.empty:
        print(f"  [Bi-monthly] 0 files found in {directory}")
        return df

    print(f"  [Bi-monthly] {len(df):3d} files  "
          f"({df.year.min()}–{df.year.max()})")
    return df


def build_quarterly_catalog(directory):
    directory = Path(directory)          # ← same fix
    files     = sorted(glob.glob(str(directory / "*.tif")))
    records   = []

    for f in files:
        stem = Path(f).stem
        year = _parse_year(stem)
        q    = _parse_quarter(stem)
        if (year is None or q is None
                or year < cfg.STUDY_START
                or year > cfg.STUDY_END):
            continue
        records.append({
            "year":    year,
            "quarter": q,
            "period":  f"{year}_Q{q}",
            "path":    f,
        })

    df = (pd.DataFrame(records)
            .sort_values(["year", "quarter"])
            .reset_index(drop=True))

    if df.empty:
        print(f"  [Quarterly]  0 files found in {directory}")
        return df

    print(f"  [Quarterly]  {len(df):3d} files  "
          f"({df.year.min()}–{df.year.max()})")
    return df


# =============================================================================
# 2.  AREAL DYNAMICS AND SEASONAL VARIABILITY  (§ 4.1)
# =============================================================================

def compute_area_series(catalog, tag):
    rows = []
    for _, row in tqdm(catalog.iterrows(),
                       total=len(catalog),
                       desc=f"  Area [{tag}]"):
        mask, _, _ = load_mask(row["path"])
        entry = row.to_dict()
        entry["area_ha"] = pixel_area_ha(mask)
        rows.append(entry)

    df  = pd.DataFrame(rows)
    out = cfg.CSV_DIR / f"{tag}_water_area.csv"
    df.to_csv(out, index=False)
    print(f"    → {out.name}  ({len(df)} records)")
    return df


def mann_kendall_trend(yearly_area):
    y = yearly_area["area_ha"].values
    x = yearly_area["year"].values

    slope, intercept, _, _ = stats.theilslopes(y, x)[:4]

    if MK_AVAILABLE:
        res   = mk.original_test(y)
        trend = res.trend
        p_val = res.p
        tau   = res.Tau
    else:
        tau, p_val = stats.kendalltau(x, y)
        if p_val < 0.05:
            trend = "increasing" if tau > 0 else "decreasing"
        else:
            trend = "no trend"

    result = {
        "trend":       trend,
        "p_value":     round(float(p_val),      4),
        "tau":         round(float(tau),         4),
        "slope_ha_yr": round(float(slope),       2),
        "intercept":   round(float(intercept),   2),
    }

    print("\n  ── Mann-Kendall Trend Test (Yearly Area) ──────────────")
    for k, v in result.items():
        print(f"     {k:<14}: {v}")

    pd.DataFrame([result]).to_csv(
        cfg.CSV_DIR / "mann_kendall_result.csv", index=False)
    return result


def seasonal_cv(bimonthly_area):
    grp = bimonthly_area.groupby("year")["area_ha"]

    cv_df = pd.DataFrame({
        "year":         grp.mean().index,
        "mean_ha":      grp.mean().values,
        "std_ha":       grp.std().values,
        "min_ha":       grp.min().values,
        "max_ha":       grp.max().values,
        "amplitude_ha": (grp.max() - grp.min()).values,
    })
    cv_df["cv_pct"] = (cv_df["std_ha"] / cv_df["mean_ha"]) * 100

    out = cfg.CSV_DIR / "seasonal_cv.csv"
    cv_df.to_csv(out, index=False)

    print(f"\n  Seasonal CV saved → {out.name}")
    print(f"     Mean CV = {cv_df.cv_pct.mean():.1f}%  "
          f"(range {cv_df.cv_pct.min():.1f}–{cv_df.cv_pct.max():.1f}%)")
    return cv_df


# ── Plots for § 4.1 ──────────────────────────────────────────────────────────

def plot_yearly_area_trend(yearly_area, mk_result):
    fig, ax = plt.subplots(figsize=cfg.FIGSIZE_WIDE)

    x       = yearly_area["year"].values
    y       = yearly_area["area_ha"].values
    trend_y = mk_result["slope_ha_yr"] * x + mk_result["intercept"]

    ax.fill_between(x, y, alpha=0.25, color="steelblue")
    ax.plot(x, y, "o-", color="steelblue", lw=2, ms=5,
            label="Annual water area")
    ax.plot(x, trend_y, "--", color="crimson", lw=1.8,
            label=(f"Theil-Sen slope "
                   f"({mk_result['slope_ha_yr']:+.0f} ha yr⁻¹)"))

    sig_str = ("significant" if mk_result["p_value"] < 0.05
               else "not significant")
    ax.text(
        0.02, 0.96,
        (f"Mann-Kendall: {mk_result['trend']}  "
         f"(τ = {mk_result['tau']},  "
         f"p = {mk_result['p_value']},  {sig_str})"),
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round,pad=0.3",
                  fc="lightyellow", ec="gray", alpha=0.9),
    )

    ax.set(
        xlabel="Year",
        ylabel="Water Area (ha)",
        title=(f"Padma River – Long-term Annual Water Area "
               f"({cfg.STUDY_START}–{cfg.STUDY_END})"),
    )
    ax.legend()
    ax.set_xlim(cfg.STUDY_START - 0.5, cfg.STUDY_END + 0.5)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
    plt.tight_layout()
    fig.savefig(cfg.FIGURES_DIR / "fig1_yearly_area_trend.png", dpi=cfg.DPI)
    plt.close(fig)
    print("  → fig1_yearly_area_trend.png")


def plot_seasonal_profiles(bimonthly_area, cv_df):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=cfg.FIGSIZE_WIDE)

    years = sorted(bimonthly_area["year"].unique())
    cmap  = plt.cm.plasma
    norm  = mcolors.Normalize(vmin=min(years), vmax=max(years))

    for yr in years:
        sub = (bimonthly_area[bimonthly_area.year == yr]
               .sort_values("bimonth"))
        ax1.plot(sub["bimonth"], sub["area_ha"],
                 color=cmap(norm(yr)), lw=1.2, alpha=0.7)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=ax1, label="Year", pad=0.02)

    ax1.set_xticks(range(1, 7))
    ax1.set_xticklabels(list(cfg.BM_LABELS.values()),
                        rotation=30, ha="right")
    ax1.set(
        xlabel="Bi-monthly Period",
        ylabel="Water Area (ha)",
        title=(f"Seasonal Water Area Profiles\n"
               f"({cfg.STUDY_START}–{cfg.STUDY_END}, all years overlaid)"),
    )

    cv_norm = mcolors.Normalize()(cv_df["cv_pct"].values)
    ax2.bar(cv_df["year"], cv_df["cv_pct"],
            color=plt.cm.RdYlGn_r(cv_norm),
            edgecolor="white", linewidth=0.4)
    ax2.axhline(cv_df["cv_pct"].mean(), color="navy",
                ls="--", lw=1.5, label="Mean CV")
    ax2.set(
        xlabel="Year",
        ylabel="Coefficient of Variation (%)",
        title="Annual Seasonal Amplitude (CV)\nof Bi-monthly Water Area",
    )
    ax2.legend()
    ax2.set_xlim(cfg.STUDY_START - 0.5, cfg.STUDY_END + 0.5)
    ax2.xaxis.set_major_locator(mticker.MultipleLocator(5))

    plt.tight_layout()
    fig.savefig(cfg.FIGURES_DIR / "fig2_seasonal_profiles_cv.png", dpi=cfg.DPI)
    plt.close(fig)
    print("  → fig2_seasonal_profiles_cv.png")


def plot_quarterly_heatmap(quarterly_area):
    pivot = quarterly_area.pivot(
        index="quarter", columns="year", values="area_ha"
    )
    fig, ax = plt.subplots(
        figsize=(max(12, len(pivot.columns) * 0.55 + 2), 4)
    )
    sns.heatmap(
        pivot, ax=ax, cmap="Blues",
        linewidths=0.3, linecolor="white",
        cbar_kws={"label": "Water Area (ha)"},
        yticklabels=["Q1 (Jan–Mar)", "Q2 (Apr–Jun)",
                     "Q3 (Jul–Sep)", "Q4 (Oct–Dec)"],
    )
    ax.set(
        title=(f"Quarterly Water Area Heat-map  (ha)  "
               f"{cfg.STUDY_START}–{cfg.STUDY_END}"),
        xlabel="Year",
        ylabel="Quarter",
    )
    plt.tight_layout()
    fig.savefig(cfg.FIGURES_DIR / "fig3_quarterly_heatmap.png", dpi=cfg.DPI)
    plt.close(fig)
    print("  → fig3_quarterly_heatmap.png")


# =============================================================================
# 3.  EROSION AND ACCRETION MAPPING  (§ 4.2)
# =============================================================================

def change_map(mask_t1, mask_t2):
    """
    ΔT1→T2 = W_T2 − W_T1
      +1 → Erosion   (Land  → Water)
       0 → Stable
      -1 → Accretion (Water → Land)
    """
    t1      = mask_t1.astype(np.int8)
    t2      = mask_t2.astype(np.int8)
    delta   = t2 - t1
    no_data = np.isnan(mask_t1) | np.isnan(mask_t2)
    delta[no_data] = 0
    return delta


def compute_annual_erosion_accretion(yearly_catalog):
    rows  = []
    irows = list(yearly_catalog.iterrows())

    for (_, r1), (_, r2) in tqdm(pairwise(irows),
                                  desc="  Erosion/Accretion [annual]",
                                  total=len(irows) - 1):
        m1, _, meta = load_mask(r1["path"])
        m2, _,  _   = load_mask(r2["path"])
        delta        = change_map(m1, m2)

        e_ha = float(np.sum(delta ==  1)) * cfg.PIXEL_AREA_HA
        a_ha = float(np.sum(delta == -1)) * cfg.PIXEL_AREA_HA
        net  = e_ha - a_ha

        rows.append({
            "from_year":    int(r1["year"]),
            "to_year":      int(r2["year"]),
            "erosion_ha":   round(e_ha,  2),
            "accretion_ha": round(a_ha,  2),
            "net_ha":       round(net,   2),
        })

        save_raster(
            delta.astype(float), meta,
            cfg.RASTER_DIR / f"change_{r1['year']}_{r2['year']}.tif",
            dtype="int8",
        )

    df = pd.DataFrame(rows)
    df.to_csv(cfg.CSV_DIR / "annual_erosion_accretion.csv", index=False)
    print(f"    → annual_erosion_accretion.csv  ({len(df)} transitions)")
    return df


def compute_decadal_erosion_accretion(yearly_catalog):
    years = yearly_catalog["year"].values

    def nearest_row(target):
        idx = int(np.argmin(np.abs(years - target)))
        return yearly_catalog.iloc[idx]

    rows   = []
    bounds = cfg.DECADAL_BOUNDARIES

    for b1, b2 in pairwise(bounds):
        r1 = nearest_row(b1)
        r2 = nearest_row(b2)
        m1, _, meta = load_mask(r1["path"])
        m2, _,  _   = load_mask(r2["path"])
        delta = change_map(m1, m2)

        rows.append({
            "period":       f"{r1['year']}–{r2['year']}",
            "from_year":    int(r1["year"]),
            "to_year":      int(r2["year"]),
            "erosion_ha":   round(np.sum(delta ==  1) * cfg.PIXEL_AREA_HA, 1),
            "accretion_ha": round(np.sum(delta == -1) * cfg.PIXEL_AREA_HA, 1),
            "net_ha":       round((np.sum(delta ==  1) -
                                   np.sum(delta == -1))
                                  * cfg.PIXEL_AREA_HA, 1),
        })

    df = pd.DataFrame(rows)
    df.to_csv(cfg.CSV_DIR / "decadal_erosion_accretion.csv", index=False)
    print(f"    → decadal_erosion_accretion.csv  ({len(df)} periods)")
    return df


def track_char_lands(yearly_catalog):
    rows             = []
    prev_centroids   = {}

    for _, row in tqdm(yearly_catalog.iterrows(),
                       total=len(yearly_catalog),
                       desc="  Char-land tracking"):
        mask, _, _ = load_mask(row["path"])
        water      = mask.astype(bool)
        land       = ~water
        land       = binary_fill_holes(land) & land

        labeled, _ = ndlabel(land)

        current_centroids = {}

        for region in regionprops(labeled):
            if region.area < cfg.MIN_ISLAND_PX:
                continue

            cy, cx  = region.centroid
            area_ha = region.area * cfg.PIXEL_AREA_HA

            migration_m = np.nan
            if prev_centroids:
                dists = {
                    iid: np.hypot(cy - pc[0], cx - pc[1])
                    for iid, pc in prev_centroids.items()
                }
                nearest_id   = min(dists, key=dists.get)
                nearest_dist = dists[nearest_id]
                if nearest_dist < 50:
                    migration_m = nearest_dist * cfg.PIXEL_SIZE_M

            rows.append({
                "year":         int(row["year"]),
                "island_id":    region.label,
                "centroid_row": round(cy, 2),
                "centroid_col": round(cx, 2),
                "area_ha":      round(area_ha, 2),
                "migration_m":  (round(float(migration_m), 1)
                                 if not np.isnan(migration_m) else np.nan),
            })
            current_centroids[region.label] = (cy, cx)

        prev_centroids = current_centroids

    df = pd.DataFrame(rows)
    df.to_csv(cfg.CSV_DIR / "char_land_tracking.csv", index=False)
    print(f"    → char_land_tracking.csv  ({len(df)} island-year records)")
    return df


# ── Plots for § 4.2 ──────────────────────────────────────────────────────────

def plot_erosion_accretion(annual_ea, decadal_ea):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=cfg.FIGSIZE_TALL)

    years  = annual_ea["from_year"].values
    erode  = annual_ea["erosion_ha"].values
    accret = annual_ea["accretion_ha"].values
    net    = annual_ea["net_ha"].values

    ax1.bar(years,  erode,  0.8, color="#d73027",
            label="Gross Erosion",   alpha=0.85)
    ax1.bar(years, -accret, 0.8, color="#4575b4",
            label="Gross Accretion", alpha=0.85)
    ax1.plot(years, net, "s-", color="black", lw=1.5, ms=4,
             label="Net Change")
    ax1.axhline(0, color="gray", lw=0.8)
    ax1.set(
        xlabel="Year",
        ylabel="Area (ha)",
        title=(f"Annual Gross Erosion, Accretion, and Net Change  "
               f"({cfg.STUDY_START}–{cfg.STUDY_END})"),
    )
    ax1.legend(ncol=3)
    ax1.set_xlim(cfg.STUDY_START - 1, cfg.STUDY_END + 1)
    ax1.xaxis.set_major_locator(mticker.MultipleLocator(5))

    x = np.arange(len(decadal_ea))
    w = 0.28
    ax2.bar(x - w, decadal_ea["erosion_ha"],   w,
            color="#d73027", label="Erosion",   alpha=0.85)
    ax2.bar(x,     decadal_ea["accretion_ha"], w,
            color="#4575b4", label="Accretion", alpha=0.85)
    ax2.bar(x + w, decadal_ea["net_ha"],       w,
            color="#1a9641", label="Net",       alpha=0.85)
    ax2.set_xticks(x)
    ax2.set_xticklabels(decadal_ea["period"], rotation=15)
    ax2.axhline(0, color="gray", lw=0.8)
    ax2.set(
        ylabel="Area (ha)",
        title="Decadal Erosion and Accretion Summary",
    )
    ax2.legend()

    plt.tight_layout()
    fig.savefig(cfg.FIGURES_DIR / "fig4_erosion_accretion.png", dpi=cfg.DPI)
    plt.close(fig)
    print("  → fig4_erosion_accretion.png")


def plot_change_map(yearly_catalog, year_t1, year_t2):
    if year_t1 < cfg.STUDY_START:
        raise ValueError(
            f"year_t1={year_t1} predates STUDY_START={cfg.STUDY_START}."
        )

    r1 = yearly_catalog[yearly_catalog.year == year_t1].iloc[0]
    r2 = yearly_catalog[yearly_catalog.year == year_t2].iloc[0]
    m1, _, _ = load_mask(r1["path"])
    m2, _, _ = load_mask(r2["path"])
    delta    = change_map(m1, m2)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, arr, title, cmap in zip(
            axes,
            [m1,   m2,   delta],
            [f"Water Mask {year_t1}",
             f"Water Mask {year_t2}",
             f"Change  {year_t1} → {year_t2}"],
            ["Blues", "Blues", cfg.CMAP_CHANGE],
    ):
        im = ax.imshow(arr, cmap=cmap, interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

    legend_elements = [
        Patch(facecolor="#d73027", label="Erosion   (+1)"),
        Patch(facecolor="#f7f7f7", label="Stable    ( 0)", ec="gray"),
        Patch(facecolor="#4575b4", label="Accretion (−1)"),
    ]
    axes[2].legend(handles=legend_elements, loc="lower right", fontsize=9)

    plt.suptitle(f"Bi-temporal Change Map:  {year_t1} → {year_t2}",
                 fontsize=14, y=1.01)
    plt.tight_layout()
    out = cfg.FIGURES_DIR / f"fig5_change_map_{year_t1}_{year_t2}.png"
    fig.savefig(out, dpi=cfg.DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  → {out.name}")


def plot_char_migration(char_df):
    annual = (
        char_df.groupby("year")
        .agg(
            total_area_ha=("area_ha",     "sum"),
            n_islands    =("island_id",   "count"),
            mean_migr_m  =("migration_m", "mean"),
        )
        .reset_index()
    )

    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

    axes[0].fill_between(annual.year, annual.total_area_ha,
                         alpha=0.4, color="sandybrown")
    axes[0].plot(annual.year, annual.total_area_ha,
                 "o-", color="saddlebrown", lw=2)
    axes[0].set_ylabel("Total Char Area (ha)")
    axes[0].set_title("Char-land (Mid-channel Bar) Dynamics  "
                      f"({cfg.STUDY_START}–{cfg.STUDY_END})")

    axes[1].bar(annual.year, annual.n_islands, color="peru", alpha=0.8)
    axes[1].set_ylabel("Number of Active Bars")

    axes[2].plot(annual.year, annual.mean_migr_m,
                 "s-", color="firebrick", lw=2)
    axes[2].set_ylabel("Mean Migration (m yr⁻¹)")
    axes[2].set_xlabel("Year")

    for ax in axes:
        ax.set_xlim(cfg.STUDY_START - 0.5, cfg.STUDY_END + 0.5)
        ax.xaxis.set_major_locator(mticker.MultipleLocator(5))

    plt.tight_layout()
    fig.savefig(cfg.FIGURES_DIR / "fig6_char_migration.png", dpi=cfg.DPI)
    plt.close(fig)
    print("  → fig6_char_migration.png")


# =============================================================================
# 4.  MORPHOLOGICAL METRICS & CHANNEL MIGRATION  (§ 4.3)
# =============================================================================

def extract_centerline(mask):
    clean = remove_small_objects(mask.astype(bool), min_size=100)
    clean = binary_fill_holes(clean)
    return skeletonize(clean)


def centerline_lateral_shift(skel1, skel2):
    dist      = distance_transform_edt(~skel1)
    shifts_px = dist[skel2]
    if shifts_px.size == 0:
        return np.nan
    return float(np.median(shifts_px)) * cfg.PIXEL_SIZE_M


def compute_centerline_migration(yearly_catalog):
    rows      = []
    prev_skel = None
    prev_year = None

    for _, row in tqdm(yearly_catalog.iterrows(),
                       total=len(yearly_catalog),
                       desc="  Centerline migration"):
        mask, _, _ = load_mask(row["path"])
        skel       = extract_centerline(mask)

        if prev_skel is not None:
            span    = row["year"] - prev_year
            shift_m = centerline_lateral_shift(prev_skel, skel)
            rows.append({
                "from_year":      int(prev_year),
                "to_year":        int(row["year"]),
                "shift_m":        (round(shift_m, 1)
                                   if not np.isnan(shift_m) else np.nan),
                "shift_m_per_yr": (round(shift_m / span, 1)
                                   if not np.isnan(shift_m) else np.nan),
            })

        prev_skel = skel
        prev_year = row["year"]

    df = pd.DataFrame(rows)
    df.to_csv(cfg.CSV_DIR / "centerline_migration.csv", index=False)
    print(f"    → centerline_migration.csv  ({len(df)} transitions)")
    return df


def compute_transect_widths(yearly_catalog):
    m0, _, _ = load_mask(yearly_catalog.iloc[0]["path"])
    H, W     = m0.shape
    half     = min(cfg.TRANSECT_HALF_LEN_PX, H // 2 - 1)
    tc_cols  = np.arange(cfg.TRANSECT_SPACING_PX,
                         W - cfg.TRANSECT_SPACING_PX,
                         cfg.TRANSECT_SPACING_PX)
    widths   = {}

    for _, row in tqdm(yearly_catalog.iterrows(),
                       total=len(yearly_catalog),
                       desc="  Transect widths"):
        mask, _, _ = load_mask(row["path"])
        col_widths = []
        for col in tc_cols:
            strip = mask[max(0, H // 2 - half): min(H, H // 2 + half), col]
            col_widths.append(float(np.nansum(strip)) * cfg.PIXEL_SIZE_M)
        widths[int(row["year"])] = col_widths

    df = pd.DataFrame(widths, index=tc_cols)
    df.index.name = "transect_col_px"
    df.to_csv(cfg.CSV_DIR / "transect_widths.csv")
    print(f"    → transect_widths.csv  "
          f"({len(tc_cols)} transects × {len(yearly_catalog)} years)")
    return df


# ── Plots for § 4.3 ──────────────────────────────────────────────────────────

def plot_centerline_migration(migr_df):
    fig, ax = plt.subplots(figsize=cfg.FIGSIZE_WIDE)

    values = migr_df["shift_m_per_yr"].fillna(0).values
    colors = ["#d73027" if v > 0 else "#4575b4" for v in values]

    ax.bar(migr_df["from_year"], values,
           color=colors, edgecolor="white", lw=0.4)
    mean_val = migr_df["shift_m_per_yr"].mean()
    ax.axhline(mean_val, color="black", ls="--", lw=1.5,
               label=f"Mean = {mean_val:.0f} m yr⁻¹")

    ax.set(
        xlabel="From Year",
        ylabel="Lateral Migration (m yr⁻¹)",
        title=(f"Padma River Centerline Lateral Migration Rate  "
               f"({cfg.STUDY_START}–{cfg.STUDY_END})"),
    )
    ax.legend()
    ax.set_xlim(cfg.STUDY_START - 1, cfg.STUDY_END + 1)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
    plt.tight_layout()
    fig.savefig(cfg.FIGURES_DIR / "fig7_centerline_migration.png", dpi=cfg.DPI)
    plt.close(fig)
    print("  → fig7_centerline_migration.png")


def plot_transect_heatmap(width_df):
    fig, ax = plt.subplots(
        figsize=(max(12, len(width_df.columns) * 0.5 + 2), 6)
    )
    sns.heatmap(
        width_df, ax=ax, cmap="Blues",
        cbar_kws={"label": "Channel Width (m)"},
        xticklabels=5,
        yticklabels=False,
    )
    ax.set(
        xlabel="Year",
        ylabel="Transect position  (upstream → downstream)",
        title=(f"Cross-sectional Channel Width Heat-map  (m)  "
               f"{cfg.STUDY_START}–{cfg.STUDY_END}"),
    )
    plt.tight_layout()
    fig.savefig(cfg.FIGURES_DIR / "fig8_transect_width_heatmap.png", dpi=cfg.DPI)
    plt.close(fig)
    print("  → fig8_transect_width_heatmap.png")


# =============================================================================
# 5.  PROBABILISTIC MAPPING  (§ 4.4)
# =============================================================================

def compute_wof(catalog, tag):
    stack = []
    meta  = None

    for _, row in tqdm(catalog.iterrows(),
                       total=len(catalog),
                       desc=f"  WOF [{tag}]"):
        mask, _, m = load_mask(row["path"])
        if meta is None:
            meta = m
        stack.append(mask.astype(np.float32))

    cube  = np.stack(stack, axis=0)
    valid = np.sum(~np.isnan(cube), axis=0)
    wof   = np.where(
        valid > 0,
        np.nansum(cube, axis=0) / valid * 100,
        np.nan,
    )

    save_raster(wof, meta, cfg.RASTER_DIR / f"wof_{tag}.tif")
    print(f"    → wof_{tag}.tif")

    flat    = wof[~np.isnan(wof)]
    perm_ha = np.sum(flat >= cfg.WOF_PERMANENT)      * cfg.PIXEL_AREA_HA
    act_ha  = np.sum((flat >= cfg.WOF_ACTIVE_LOW) &
                     (flat <  cfg.WOF_PERMANENT))    * cfg.PIXEL_AREA_HA
    eph_ha  = np.sum(flat < cfg.WOF_ACTIVE_LOW)      * cfg.PIXEL_AREA_HA

    print(f"     Permanent channel  (WOF ≥ {cfg.WOF_PERMANENT}%) : "
          f"{perm_ha:>10,.0f} ha")
    print(f"     Active corridor    ({cfg.WOF_ACTIVE_LOW}–"
          f"{cfg.WOF_PERMANENT}%)       : {act_ha:>10,.0f} ha")
    print(f"     Ephemeral / land   (WOF < {cfg.WOF_ACTIVE_LOW}%) : "
          f"{eph_ha:>10,.0f} ha")

    pd.DataFrame([{
        "tag":          tag,
        "permanent_ha": round(perm_ha, 1),
        "active_ha":    round(act_ha,  1),
        "ephemeral_ha": round(eph_ha,  1),
    }]).to_csv(cfg.CSV_DIR / f"wof_zones_{tag}.csv", index=False)

    return wof, meta


def compute_markov_transitions(catalog, tag):
    count  = np.zeros((2, 2), dtype=np.float64)
    H, W   = _shape_from_catalog(catalog)
    sp_cnt = np.zeros((2, 2, H, W), dtype=np.float32)

    prev_mask = None
    prev_meta = None

    for _, row in tqdm(catalog.iterrows(),
                       total=len(catalog),
                       desc=f"  Markov [{tag}]"):
        mask, _, meta = load_mask(row["path"])
        if prev_mask is None:
            prev_mask = mask
            prev_meta = meta
            continue

        valid = ~np.isnan(prev_mask) & ~np.isnan(mask)
        t1    = prev_mask.astype(np.int8)
        t2    = mask.astype(np.int8)

        for fi in (0, 1):
            for ti in (0, 1):
                idx              = valid & (t1 == fi) & (t2 == ti)
                count[fi, ti]   += idx.sum()
                sp_cnt[fi, ti]  += idx.astype(np.float32)

        prev_mask = mask
        prev_meta = meta

    row_sums = count.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    prob    = count / row_sums
    labels  = ["Land (0)", "Water (1)"]
    df_prob = pd.DataFrame(prob, index=labels, columns=labels)
    df_prob.index.name = "From \\ To"
    df_prob.to_csv(cfg.CSV_DIR / f"markov_matrix_{tag}.csv")

    print(f"\n    Markov Transition Matrix  [{tag}]")
    print(df_prob.to_string())

    sp_dir = cfg.RASTER_DIR / f"spatial_transitions_{tag}"
    sp_dir.mkdir(exist_ok=True)
    names  = {
        (0, 0): "LL_land2land",
        (0, 1): "LW_land2water",
        (1, 0): "WL_water2land",
        (1, 1): "WW_water2water",
    }
    for (fi, ti), name in names.items():
        total = sp_cnt[fi, 0] + sp_cnt[fi, 1]
        total[total == 0] = np.nan
        sp_prob = sp_cnt[fi, ti] / total * 100
        save_raster(sp_prob, prev_meta, sp_dir / f"{name}.tif")

    print(f"    → spatial transition rasters → "
          f"{sp_dir.relative_to(cfg.OUTPUT_DIR)}/")
    return df_prob


# ── Plots for § 4.4 ──────────────────────────────────────────────────────────

def plot_wof_map(wof, tag):
    fig, ax = plt.subplots(figsize=cfg.FIGSIZE_MAP)

    im = ax.imshow(wof, cmap=cfg.CMAP_WOF,
                   vmin=0, vmax=100, interpolation="bilinear")
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02,
                 label="Water Occurrence Frequency (%)")

    for thresh, ls, col in [
        (cfg.WOF_PERMANENT,  "--", "navy"),
        (cfg.WOF_ACTIVE_LOW, ":",  "darkred"),
    ]:
        ax.contour(wof, levels=[thresh], colors=[col],
                   linestyles=[ls], linewidths=[1.2])

    legend_elements = [
        Line2D([0], [0], color="navy",    ls="--", lw=1.5,
               label=f"Permanent core  (≥ {cfg.WOF_PERMANENT}%)"),
        Line2D([0], [0], color="darkred", ls=":",  lw=1.5,
               label=f"Active corridor (≥ {cfg.WOF_ACTIVE_LOW}%)"),
    ]
    ax.legend(handles=legend_elements, loc="lower right")
    ax.set_title(f"Water Occurrence Frequency  [{tag}]  "
                 f"({cfg.STUDY_START}–{cfg.STUDY_END})")
    ax.axis("off")
    plt.tight_layout()
    fig.savefig(cfg.FIGURES_DIR / f"fig9_wof_{tag}.png", dpi=cfg.DPI)
    plt.close(fig)
    print(f"  → fig9_wof_{tag}.png")


def plot_wof_histogram(wof, tag):
    flat = wof[~np.isnan(wof)]
    fig, ax = plt.subplots(figsize=(9, 4))

    ax.hist(flat, bins=50, color="steelblue", edgecolor="white",
            linewidth=0.4, density=True, label="WOF distribution")

    ax.axvspan(0,                  cfg.WOF_ACTIVE_LOW, alpha=0.12,
               color="gray",   label="Ephemeral / land")
    ax.axvspan(cfg.WOF_ACTIVE_LOW, cfg.WOF_PERMANENT,  alpha=0.12,
               color="orange", label="Active corridor")
    ax.axvspan(cfg.WOF_PERMANENT,  100,                alpha=0.12,
               color="navy",   label="Permanent channel")

    ax.axvline(cfg.WOF_ACTIVE_LOW, color="darkred", ls=":", lw=1.5)
    ax.axvline(cfg.WOF_PERMANENT,  color="navy",    ls="--", lw=1.5)

    ax.set(
        xlabel="Water Occurrence Frequency (%)",
        ylabel="Density",
        title=f"WOF Distribution  [{tag}]  "
              f"({cfg.STUDY_START}–{cfg.STUDY_END})",
    )
    ax.legend()
    plt.tight_layout()
    fig.savefig(cfg.FIGURES_DIR / f"fig10_wof_histogram_{tag}.png", dpi=cfg.DPI)
    plt.close(fig)
    print(f"  → fig10_wof_histogram_{tag}.png")


def plot_markov_heatmap(df_prob, tag):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        df_prob, annot=True, fmt=".3f",
        cmap="YlOrRd", vmin=0, vmax=1,
        linewidths=0.5, ax=ax,
        cbar_kws={"label": "Transition Probability"},
    )
    ax.set_title(f"Markov Transition Matrix  [{tag}]  "
                 f"({cfg.STUDY_START}–{cfg.STUDY_END})")
    ax.set_xlabel("To State")
    ax.set_ylabel("From State")
    plt.tight_layout()
    out = cfg.FIGURES_DIR / f"fig11_markov_{tag}.png"
    fig.savefig(out, dpi=cfg.DPI)
    plt.close(fig)
    print(f"  → {out.name}")


# =============================================================================
# 6.  SUMMARY REPORT
# =============================================================================

def print_summary_report(yearly_area, mk_result, cv_df,
                          annual_ea,   migr_df):
    sep = "─" * 60

    print(f"\n{'=' * 60}")
    print("  PADMA RIVER MORPHODYNAMICS – SUMMARY REPORT")
    print(f"  Study period : {cfg.STUDY_START}–{cfg.STUDY_END}  "
          f"(1987 excluded – broken imagery)")
    print(f"{'=' * 60}")

    # § 4.1
    print(f"\n{sep}")
    print("  § 4.1  AREAL DYNAMICS")
    print(sep)
    print(f"  Study period          : {int(yearly_area['year'].min())}–"
          f"{int(yearly_area['year'].max())}")
    print(f"  Total years analysed  : {len(yearly_area)}")
    print(f"  Mean annual area      : {yearly_area['area_ha'].mean():>12,.1f} ha")
    print(f"  Std  annual area      : {yearly_area['area_ha'].std():>12,.1f} ha")
    print(f"  Min  annual area      : {yearly_area['area_ha'].min():>12,.1f} ha  "
          f"(year {int(yearly_area.loc[yearly_area['area_ha'].idxmin(), 'year'])})")
    print(f"  Max  annual area      : {yearly_area['area_ha'].max():>12,.1f} ha  "
          f"(year {int(yearly_area.loc[yearly_area['area_ha'].idxmax(), 'year'])})")
    print(f"\n  Mann-Kendall result")
    print(f"    Trend               : {mk_result['trend']}")
    print(f"    Kendall τ           : {mk_result['tau']}")
    print(f"    p-value             : {mk_result['p_value']}")
    sig = "✓ Significant" if mk_result["p_value"] < 0.05 else "✗ Not significant"
    print(f"    Significance (α=0.05): {sig}")
    print(f"    Theil-Sen slope     : {mk_result['slope_ha_yr']:>+10.1f} ha yr⁻¹")
    print(f"\n  Seasonal variability (bi-monthly CV)")
    print(f"    Mean annual CV      : {cv_df['cv_pct'].mean():>10.1f} %")
    print(f"    Min  annual CV      : {cv_df['cv_pct'].min():>10.1f} %  "
          f"(year {int(cv_df.loc[cv_df['cv_pct'].idxmin(), 'year'])})")
    print(f"    Max  annual CV      : {cv_df['cv_pct'].max():>10.1f} %  "
          f"(year {int(cv_df.loc[cv_df['cv_pct'].idxmax(), 'year'])})")
    print(f"    Mean amplitude      : {cv_df['amplitude_ha'].mean():>10,.1f} ha")
    print(f"    Max  amplitude      : {cv_df['amplitude_ha'].max():>10,.1f} ha  "
          f"(year {int(cv_df.loc[cv_df['amplitude_ha'].idxmax(), 'year'])})")

    # § 4.2
    print(f"\n{sep}")
    print("  § 4.2  EROSION AND ACCRETION")
    print(sep)
    print(f"  Transitions analysed  : {len(annual_ea)}")
    print(f"  Period                : "
          f"{int(annual_ea['from_year'].min())}–"
          f"{int(annual_ea['to_year'].max())}")
    print(f"\n  Annual averages")
    print(f"    Gross erosion       : {annual_ea['erosion_ha'].mean():>10,.1f} ha yr⁻¹")
    print(f"    Gross accretion     : {annual_ea['accretion_ha'].mean():>10,.1f} ha yr⁻¹")
    print(f"    Net change          : {annual_ea['net_ha'].mean():>+10,.1f} ha yr⁻¹")
    max_e_idx = annual_ea['erosion_ha'].idxmax()
    max_a_idx = annual_ea['accretion_ha'].idxmax()
    max_n_idx = annual_ea['net_ha'].abs().idxmax()
    print(f"\n  Extreme years")
    print(f"    Peak erosion        : "
          f"{annual_ea.loc[max_e_idx, 'erosion_ha']:>10,.1f} ha  "
          f"({int(annual_ea.loc[max_e_idx, 'from_year'])}–"
          f"{int(annual_ea.loc[max_e_idx, 'to_year'])})")
    print(f"    Peak accretion      : "
          f"{annual_ea.loc[max_a_idx, 'accretion_ha']:>10,.1f} ha  "
          f"({int(annual_ea.loc[max_a_idx, 'from_year'])}–"
          f"{int(annual_ea.loc[max_a_idx, 'to_year'])})")
    print(f"    Peak net change     : "
          f"{annual_ea.loc[max_n_idx, 'net_ha']:>+10,.1f} ha  "
          f"({int(annual_ea.loc[max_n_idx, 'from_year'])}–"
          f"{int(annual_ea.loc[max_n_idx, 'to_year'])})")
    total_e = annual_ea['erosion_ha'].sum()
    total_a = annual_ea['accretion_ha'].sum()
    print(f"\n  Cumulative totals ({cfg.STUDY_START}–{cfg.STUDY_END})")
    print(f"    Total erosion       : {total_e:>12,.1f} ha")
    print(f"    Total accretion     : {total_a:>12,.1f} ha")
    print(f"    Cumulative net      : {total_e - total_a:>+12,.1f} ha")
    print(f"    Erosion/Accretion   : {total_e / total_a:>12.3f}  "
          f"({'erosion-dominated' if total_e > total_a else 'accretion-dominated'})")

    # § 4.3
    print(f"\n{sep}")
    print("  § 4.3  CHANNEL MIGRATION")
    print(sep)
    valid_m = migr_df.dropna(subset=["shift_m_per_yr"])
    if len(valid_m) > 0:
        print(f"  Transitions with valid shift : {len(valid_m)}")
        print(f"  Mean lateral migration       : "
              f"{valid_m['shift_m_per_yr'].mean():>8,.1f} m yr⁻¹")
        print(f"  Median lateral migration     : "
              f"{valid_m['shift_m_per_yr'].median():>8,.1f} m yr⁻¹")
        print(f"  Std  lateral migration       : "
              f"{valid_m['shift_m_per_yr'].std():>8,.1f} m yr⁻¹")
        max_m_idx = valid_m["shift_m_per_yr"].idxmax()
        min_m_idx = valid_m["shift_m_per_yr"].idxmin()
        print(f"  Max migration                : "
              f"{valid_m.loc[max_m_idx, 'shift_m_per_yr']:>8,.1f} m yr⁻¹  "
              f"({int(valid_m.loc[max_m_idx, 'from_year'])}–"
              f"{int(valid_m.loc[max_m_idx, 'to_year'])})")
        print(f"  Min migration                : "
              f"{valid_m.loc[min_m_idx, 'shift_m_per_yr']:>8,.1f} m yr⁻¹  "
              f"({int(valid_m.loc[min_m_idx, 'from_year'])}–"
              f"{int(valid_m.loc[min_m_idx, 'to_year'])})")
        total_km = valid_m["shift_m_per_yr"].sum() / 1000
        print(f"  Cumulative displacement      : {total_km:>8.2f} km")
    else:
        print("  No valid migration data available.")

    print(f"\n{'=' * 60}")
    print("  All outputs written to:")
    print(f"    Figures  → {cfg.FIGURES_DIR}")
    print(f"    CSVs     → {cfg.CSV_DIR}")
    print(f"    Rasters  → {cfg.RASTER_DIR}")
    print(f"{'=' * 60}\n")


# =============================================================================
# 7.  MAIN ORCHESTRATION
# =============================================================================

def main():
    """
    Execute the full morphodynamics analysis pipeline in section order.

    Sections run:
      § 4.1 – Areal Dynamics and Seasonal Variability
      § 4.2 – Spatial Change Analysis: Erosion and Accretion
      § 4.3 – Morphological Metrics and Channel Migration
      § 4.4 – Probabilistic Mapping (WOF + Markov)
    """

    # ── Catalogue all data ────────────────────────────────────────────────────
    print("\n── Cataloguing data ────────────────────────────────────────")
    yearly_cat    = build_yearly_catalog(cfg.YEARLY_DIR)
    bimonthly_cat = build_bimonthly_catalog(cfg.BIMONTHLY_DIR)
    quarterly_cat = build_quarterly_catalog(cfg.QUARTERLY_DIR)

    # Guard: confirm catalogs are non-empty before proceeding
    for name, cat in [("yearly",    yearly_cat),
                      ("bimonthly", bimonthly_cat),
                      ("quarterly", quarterly_cat)]:
        if cat.empty:
            raise FileNotFoundError(
                f"No files found for '{name}' catalog.  "
                f"Check the corresponding directory path in Config."
            )

    # Safety check – confirm 1987 is absent
    if 1987 in yearly_cat["year"].values:
        raise RuntimeError(
            "1987 image found in catalog despite STUDY_START=1988. "
            "Please remove or rename the broken 1987 file and re-run."
        )

    # =========================================================================
    # § 4.1  AREAL DYNAMICS AND SEASONAL VARIABILITY
    # =========================================================================
    print("\n── § 4.1  Areal Dynamics ───────────────────────────────────")

    yearly_area    = compute_area_series(yearly_cat,    "yearly")
    bimonthly_area = compute_area_series(bimonthly_cat, "bimonthly")
    quarterly_area = compute_area_series(quarterly_cat, "quarterly")

    mk_result = mann_kendall_trend(yearly_area)
    cv_df     = seasonal_cv(bimonthly_area)

    plot_yearly_area_trend(yearly_area, mk_result)
    plot_seasonal_profiles(bimonthly_area, cv_df)
    plot_quarterly_heatmap(quarterly_area)

    # =========================================================================
    # § 4.2  EROSION AND ACCRETION MAPPING
    # =========================================================================
    print("\n── § 4.2  Erosion and Accretion ────────────────────────────")

    annual_ea  = compute_annual_erosion_accretion(yearly_cat)
    decadal_ea = compute_decadal_erosion_accretion(yearly_cat)
    char_df    = track_char_lands(yearly_cat)

    plot_erosion_accretion(annual_ea, decadal_ea)
    plot_char_migration(char_df)

    # Bi-temporal change maps – four representative decadal pairs
    representative_pairs = [
        (1988, 1998),
        (1998, 2008),
        (2008, 2018),
        (2018, 2025),
    ]
    avail = yearly_cat["year"].values
    for y1, y2 in representative_pairs:
        if y1 in avail and y2 in avail:
            plot_change_map(yearly_cat, y1, y2)
        else:
            print(f"  Skipping change map {y1}→{y2} "
                  f"(one or both years not in catalog)")

    # =========================================================================
    # § 4.3  MORPHOLOGICAL METRICS AND CHANNEL MIGRATION
    # =========================================================================
    print("\n── § 4.3  Morphological Metrics ────────────────────────────")

    migr_df  = compute_centerline_migration(yearly_cat)
    width_df = compute_transect_widths(yearly_cat)

    plot_centerline_migration(migr_df)
    plot_transect_heatmap(width_df)

    # =========================================================================
    # § 4.4  PROBABILISTIC MAPPING
    # =========================================================================
    print("\n── § 4.4  Probabilistic Mapping ────────────────────────────")

    # WOF – all three temporal resolutions
    print("\n  Computing WOF from yearly composites …")
    wof_yearly,    meta_yearly = compute_wof(yearly_cat,    "yearly")

    print("\n  Computing WOF from bi-monthly composites …")
    wof_bimonthly, meta_bm    = compute_wof(bimonthly_cat, "bimonthly")

    print("\n  Computing WOF from quarterly composites …")
    wof_quarterly, meta_qtr   = compute_wof(quarterly_cat, "quarterly")

    # Markov transition matrices
    print("\n  Computing Markov transitions from yearly composites …")
    markov_yearly    = compute_markov_transitions(yearly_cat,    "yearly")

    print("\n  Computing Markov transitions from quarterly composites …")
    markov_quarterly = compute_markov_transitions(quarterly_cat, "quarterly")

    print("\n  Computing Markov transitions from bi-monthly composites …")
    markov_bimonthly = compute_markov_transitions(bimonthly_cat, "bimonthly")

    # WOF plots
    for wof, tag in [
        (wof_yearly,    "yearly"),
        (wof_bimonthly, "bimonthly"),
        (wof_quarterly, "quarterly"),
    ]:
        plot_wof_map(wof, tag)
        plot_wof_histogram(wof, tag)

    # Markov plots
    for df_prob, tag in [
        (markov_yearly,    "yearly"),
        (markov_quarterly, "quarterly"),
        (markov_bimonthly, "bimonthly"),
    ]:
        plot_markov_heatmap(df_prob, tag)

    # =========================================================================
    # SUMMARY REPORT
    # =========================================================================
    print_summary_report(
        yearly_area = yearly_area,
        mk_result   = mk_result,
        cv_df       = cv_df,
        annual_ea   = annual_ea,
        migr_df     = migr_df,
    )

    print("Pipeline complete.  All outputs written successfully.\n")


# =============================================================================
# ENTRY POINT  ← this was the missing call in the original script
# =============================================================================
if __name__ == "__main__":
    main()

  Padma River Morphodynamics Pipeline  –  initialised
  Study period : 1988–2025  (1987 excluded – broken imagery)

── Cataloguing data ────────────────────────────────────────
  [Yearly]      38 files  (1988–2025)
               Skipped years before 1988: [1987]
  [Bi-monthly] 228 files  (1988–2025)
  [Quarterly]  152 files  (1988–2025)

── § 4.1  Areal Dynamics ───────────────────────────────────


  Area [yearly]: 100%|██████████| 38/38 [00:02<00:00, 16.98it/s]


    → yearly_water_area.csv  (38 records)


  Area [bimonthly]: 100%|██████████| 228/228 [00:13<00:00, 16.44it/s]


    → bimonthly_water_area.csv  (228 records)


...


    Markov Transition Matrix  [yearly]
           Land (0)  Water (1)
From \ To                     
Land (0)   0.993885   0.006115
Water (1)  0.215743   0.784257
    → spatial transition rasters → rasters/spatial_transitions_yearly/

  Computing Markov transitions from quarterly composites …


  Markov [quarterly]: 100%|██████████| 152/152 [00:21<00:00,  6.93it/s]



    Markov Transition Matrix  [quarterly]
           Land (0)  Water (1)
From \ To                     
Land (0)   0.987711   0.012289
Water (1)  0.303151   0.696849
    → spatial transition rasters → rasters/spatial_transitions_quarterly/

  Computing Markov transitions from bi-monthly composites …


  Markov [bimonthly]: 100%|██████████| 228/228 [00:32<00:00,  6.95it/s]



    Markov Transition Matrix  [bimonthly]
           Land (0)  Water (1)
From \ To                     
Land (0)   0.989906   0.010094
Water (1)  0.254593   0.745407
    → spatial transition rasters → rasters/spatial_transitions_bimonthly/
  → fig9_wof_yearly.png
  → fig10_wof_histogram_yearly.png
  → fig9_wof_bimonthly.png
  → fig10_wof_histogram_bimonthly.png
  → fig9_wof_quarterly.png
  → fig10_wof_histogram_quarterly.png
  → fig11_markov_yearly.png
  → fig11_markov_quarterly.png
  → fig11_markov_bimonthly.png

  PADMA RIVER MORPHODYNAMICS – SUMMARY REPORT
  Study period : 1988–2025  (1987 excluded – broken imagery)

────────────────────────────────────────────────────────────
  § 4.1  AREAL DYNAMICS
────────────────────────────────────────────────────────────
  Study period          : 1988–2025
  Total years analysed  : 38
  Mean annual area      :     66,445.4 ha
  Std  annual area      :     11,281.4 ha
  Min  annual area      :     54,038.9 ha  (year 1995)
  Max  annual area   

In [7]:
import zipfile
import os
from pathlib import Path
from IPython.display import FileLink, display

def zip_outputs(output_dir: Path = cfg.OUTPUT_DIR,
                zip_name: str = "padma_morphodynamics_outputs.zip") -> Path:
    """
    Recursively zip everything inside output_dir and save the archive
    to /kaggle/working/ for easy download.
    """
    zip_path = Path("/kaggle/working") / zip_name

    all_files = [f for f in output_dir.rglob("*") if f.is_file()]

    print(f"Zipping {len(all_files)} files from '{output_dir}' ...")

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file in all_files:
            arcname = file.relative_to(output_dir.parent)  # preserve folder structure
            zf.write(file, arcname)
            print(f"  + {arcname}")

    size_mb = zip_path.stat().st_size / (1024 ** 2)
    print(f"\n✓ Archive created : {zip_path}")
    print(f"  Size            : {size_mb:.2f} MB")
    print(f"  Files included  : {len(all_files)}")

    return zip_path


# ── Run and display a clickable download link ─────────────────────────────────
zip_path = zip_outputs()
display(FileLink(str(zip_path), result_html_prefix="⬇️  Download: "))

Zipping 88 files from '/kaggle/working/outputs' ...
  + outputs/csv/wof_zones_bimonthly.csv
  + outputs/csv/markov_matrix_quarterly.csv
  + outputs/csv/markov_matrix_yearly.csv
  + outputs/csv/markov_matrix_bimonthly.csv
  + outputs/csv/quarterly_water_area.csv
  + outputs/csv/char_land_tracking.csv
  + outputs/csv/yearly_water_area.csv
  + outputs/csv/mann_kendall_result.csv
  + outputs/csv/annual_erosion_accretion.csv
  + outputs/csv/transect_widths.csv
  + outputs/csv/seasonal_cv.csv
  + outputs/csv/decadal_erosion_accretion.csv
  + outputs/csv/wof_zones_yearly.csv
  + outputs/csv/bimonthly_water_area.csv
  + outputs/csv/wof_zones_quarterly.csv
  + outputs/csv/centerline_migration.csv
  + outputs/figures/fig9_wof_yearly.png
  + outputs/figures/fig5_change_map_2008_2018.png
  + outputs/figures/fig11_markov_yearly.png
  + outputs/figures/fig10_wof_histogram_yearly.png
  + outputs/figures/fig2_seasonal_profiles_cv.png
  + outputs/figures/fig6_char_migration.png
  + outputs/figures/fig9

/kaggle/working/padma_morphodynamics_outputs.zip